# **Math Tutor — Ukrainian Math Problem Generator (RAG + Multi-Agent)**

## **Import**


In [74]:
import os, json, random, re
from dataclasses import dataclass
from typing import Optional
import sympy as sp
import logging

import glob
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import time, urllib.request, urllib.parse


for name in ["tornado", "tornado.access", "tornado.application", "tornado.general"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)

In [53]:
random.seed(42)

GEMINI_MODEL = "gemini-3.5-flash"

if not os.getenv("GEMINI_API_KEY") or not os.getenv("WOLFRAM_APP_ID"):
    raise RuntimeError(
        "Потрібен GEMINI_API_KEY та WOLFRAM_APP_ID"
    )

TEACHER_MODEL = SOLVER_MODEL = GEMINI_MODEL
PROVIDER = "gemini"


TOPICS = ["Лінійні рівняння", "Квадратні рівняння", "Похідні",
          "Площа прямокутника", "Теорема Піфагора", "Класична ймовірність"]

## **RAG-based**

In [54]:
class TfidfRetriever:
    """Статичний RAG: TF-IDF над корпусом текстових файлів."""
    def __init__(self, folder="context"):
        self.chunks, self.sources = [], []
        for path in sorted(glob.glob(f"{folder}/*.txt")):
            text = open(path, encoding="utf-8").read()
            for sent in re.split(r"(?<=[.])\s+|\n+", text):
                s = sent.strip()
                if len(s) > 25:
                    self.chunks.append(s); self.sources.append(os.path.basename(path))
        self.vec = TfidfVectorizer()
        self.matrix = self.vec.fit_transform(self.chunks)

    def retrieve(self, query, k=2):
        sims = cosine_similarity(self.vec.transform([query]), self.matrix)[0]
        idx = sims.argsort()[::-1][:k]
        return [(self.chunks[i], self.sources[i], float(sims[i])) for i in idx if sims[i] > 0]

def wikipedia_context(topic: str) -> Optional[str]:
    """Динамічний контекст із Wikipedia (uk). None, якщо недоступно."""
    try:
        import urllib.request, urllib.parse
        title = urllib.parse.quote(topic.replace(" ", "_"))
        url = f"https://uk.wikipedia.org/api/rest_v1/page/summary/{title}"
        with urllib.request.urlopen(url, timeout=6) as r:
            data = json.loads(r.read().decode("utf-8"))
        return data.get("extract") or None
    except Exception:
        return None

def get_context(topic: str, dynamic: bool) -> str:
    if dynamic:
        ctx = wikipedia_context(topic)
        if ctx:
            return f"[Wikipedia] {ctx}"
    hits = RETRIEVER.retrieve(topic, k=2)
    if hits:
        return "[Підручник] " + " ".join(h[0] for h in hits)
    return "[Підручник] (релевантний контекст не знайдено)"

In [55]:
RETRIEVER = TfidfRetriever("context")
print("Чанків у корпусі:", len(RETRIEVER.chunks))

for t in TOPICS:
    h = RETRIEVER.retrieve(t, k=1)
    src, sim = (h[0][1], round(h[0][2],2)) if h else ("—", 0)
    print(f"  {t:24} -> {src} (sim={sim})")

Чанків у корпусі: 71
  Лінійні рівняння         -> algebra_context.txt (sim=0.58)
  Квадратні рівняння       -> algebra_context.txt (sim=0.54)
  Похідні                  -> calculus_context.txt (sim=0.58)
  Площа прямокутника       -> geometry_context.txt (sim=0.49)
  Теорема Піфагора         -> geometry_context.txt (sim=0.64)
  Класична ймовірність     -> probability_context.txt (sim=0.64)


## **SymPy Task Evaluation functions**

In [56]:
x = sp.symbols("x")

@dataclass
class Problem:
    topic: str
    kind: str
    statement: str
    answer_text: str
    truth: object
    truth_kind: str
    choices: Optional[list] = None
    correct_letter: Optional[str] = None
    formal: Optional[tuple] = None

def _roots_text(rs):
    rs = sorted(rs, key=lambda v: float(v))
    return "x in {" + ", ".join(str(r) for r in rs) + "}"

def gen(topic: str) -> Problem:
    if topic == "Лінійні рівняння":
        a = random.choice([2,3,4,5]); root = random.randint(-5,5); b = random.randint(-9,9)
        c = a*root + b
        return Problem(topic,"task",f"Розв'яжіть рівняння {a}*x + ({b}) = {c}.",
                       f"x = {sp.Rational(c-b,a)}", sp.Rational(c-b,a), "expr",
                       formal=("linear", a, b, c))
    if topic == "Квадратні рівняння":
        r1, r2 = sorted(random.sample(range(-6,7), 2))
        p, q = -(r1+r2), r1*r2
        return Problem(topic,"task",f"Розв'яжіть рівняння x^2 + ({p})*x + ({q}) = 0.",
                       _roots_text([r1,r2]), {sp.Integer(r1),sp.Integer(r2)}, "roots",
                       formal=("poly", p, q))
    if topic == "Похідні":
        a,b,c = random.randint(1,5),random.randint(1,5),random.randint(1,5)
        f = a*x**3 + b*x**2 + c*x; d = sp.diff(f,x)
        return Problem(topic,"task",f"Знайдіть похідну функції f(x) = {sp.printing.sstr(f)}.",
                       f"f'(x) = {sp.printing.sstr(d)}", d, "expr",
                       formal=("diff", f))
    if topic == "Інтеграли":
        a,b = random.randint(1,4), random.randint(1,5)
        f = a*x + b; F = sp.integrate(f,x)
        return Problem(topic,"task",f"Обчисліть невизначений інтеграл (інтеграл від {sp.printing.sstr(f)} dx).",
                       f"{sp.printing.sstr(F)} + C", F, "expr")
    if topic == "Системи рівнянь":
        xs, ys = random.randint(-4,4), random.randint(-4,4)
        a1,b1 = random.randint(1,3),random.randint(1,3)
        a2,b2 = random.randint(1,3),random.randint(-3,-1)
        c1, c2 = a1*xs+b1*ys, a2*xs+b2*ys
        stmt = f"Розв'яжіть систему: {a1}x + {b1}y = {c1};  {a2}x + ({b2})y = {c2}."
        return Problem(topic,"task",stmt,f"x = {xs}, y = {ys}",
                       {sp.Integer(xs),sp.Integer(ys)}, "roots")
    if topic == "Відсотки":
        base = random.choice([200,400,500,800,1000]); p = random.choice([5,10,15,20,25])
        val = sp.Rational(base*p,100)
        return Problem(topic,"task",f"Знайдіть {p}% від числа {base} (відповідь = шукане число).",
                       f"шукане число = {val}", val, "expr")
    if topic == "Площа прямокутника":
        a, b = random.randint(2,12), random.randint(2,12); S = a*b
        return Problem(topic,"task",f"Знайдіть площу прямокутника зі сторонами {a} і {b}.",
                       f"S = {S}", sp.Integer(S), "expr",
                       formal=("area", a, b))
    if topic == "Теорема Піфагора":
        a,b,c = random.choice([(3,4,5),(6,8,10),(5,12,13),(8,15,17),(9,12,15),(7,24,25)])
        return Problem(topic,"task",
                       f"Знайдіть гіпотенузу прямокутного трикутника з катетами {a} і {b}.",
                       f"c = {c}", sp.Integer(c), "expr",
                       formal=("pyth", a, b))
    if topic == "Класична ймовірність":
        qt = random.choice(["even","gt","div3"])
        if qt == "even":
            stmt = "Гральний кубик. Яка ймовірність випадіння парного числа?"; m = 3
        elif qt == "gt":
            k = random.randint(1,4)
            stmt = f"Гральний кубик. Яка ймовірність випадіння числа, більшого за {k}?"; m = 6-k
        else:
            stmt = "Гральний кубик. Яка ймовірність випадіння числа, кратного 3?"; m = 2
        val = sp.Rational(m,6)
        return Problem(topic,"task",stmt,f"P = {val}", val, "expr",
                       formal=("prob", m, 6))
    raise ValueError(topic)

def _perturb(p: Problem) -> str:
    if p.topic == "Класична ймовірність":
        opts = [sp.Rational(n, 6) for n in range(1, 6) if sp.Rational(n, 6) != p.truth]
        return f"P = {random.choice(opts)}"
    if p.truth_kind == "roots" and isinstance(p.truth,set) and all(isinstance(t,sp.Integer) for t in p.truth):
        rs = [int(t)+random.choice([-2,-1,1,2]) for t in sorted(p.truth,key=lambda v:float(v))]
        return _roots_text(rs)
    try:
        val = sp.nsimplify(p.truth)
        return p.answer_text.replace(str(val), str(val + random.choice([-2,-1,1,2])))
    except Exception:
        return p.answer_text + " (?)"

def to_quiz(p: Problem) -> Problem:
    letters = ["А","Б","В","Г"]; choices = [p.answer_text]; seen = {p.answer_text}; t=0
    while len(choices) < 4 and t < 40:
        t += 1; cand = _perturb(p)
        if cand not in seen: seen.add(cand); choices.append(cand)
    random.shuffle(choices)
    letter = letters[choices.index(p.answer_text)]
    body = "\n".join(f"{letters[i]}) {c}" for i,c in enumerate(choices))
    stmt = p.statement + "\nОберіть правильний варіант:\n" + body
    return Problem(p.topic,"quiz",stmt,p.answer_text,p.truth,p.truth_kind,choices,letter,p.formal)

def answer_token(r):
    return r["expected_answer"].split(")")[0] if r["type"]=="quiz" else r["expected_answer"]

In [57]:
d = gen("Квадратні рівняння"); print(d.statement, "->", d.answer_text)

Розв'яжіть рівняння x^2 + (1)*x + (-20) = 0. -> x in {-5, 4}


## **LLM Call Utils & Judge Agent**

In [58]:
def call_llm(model: str, system: str, user: str) -> str:
    """Виклик Gemini. Потрібен GEMINI_API_KEY. Повертає '' при будь-якій проблемі."""
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        r = genai.GenerativeModel(model, system_instruction=system).generate_content(user)
        return (r.text or "").strip()
    except Exception as e:
        print(f"Помилка виклику Gemini ({model}): {e}")
        return ""

def parse_md(text: str) -> dict:
    """Розбиває відповідь моделі на блоки за заголовками '## ...'."""
    out, cur = {}, None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("#"):
            cur = s.lstrip("# ").strip().upper()
            out[cur] = ""
        elif cur is not None:
            out[cur] += line + "\n"
    return {k: v.strip() for k, v in out.items()}

In [35]:
_parse = lambda s: sp.sympify(s.replace("^","**"), locals={"x":x}, rational=True)

def sympy_verify(p: Problem, text: str) -> bool:
    """Judge Експерименту 2: символьна еквівалентність із ключем (truth)."""
    try:
        tail = text.split("Відповідь:")[-1] if "Відповідь:" in text else text.splitlines()[-1]
        if p.truth_kind == "roots":
            cand = set()
            for n in re.findall(r"[-+]?\d+/?\d*", tail):
                try: cand.add(sp.nsimplify(sp.Rational(n)))
                except Exception: pass
            exp = {sp.nsimplify(t) for t in p.truth}
            return exp.issubset(cand)
        cand = tail.split("=")[-1].replace("+ C","").replace("+C","").strip()
        truth = p.truth if isinstance(p.truth, sp.Basic) else _parse(str(p.truth))
        return sp.simplify(_parse(cand) - truth) == 0
    except Exception:
        return False

In [59]:
pr = gen("Квадратні рівняння")
roots = sorted(int(t) for t in pr.truth)
alt = f"Відповідь: {roots[1]}, {roots[0]}"

print("Ключ:", pr.answer_text)
print("SymPy", repr(alt.split('Відповідь: ')[-1]), "->", sympy_verify(pr, alt))

Ключ: x in {-6, 5}
SymPy '5, -6' -> True


## **Experiment 1 — Baseline (Teacher + Judge via Wolfram Alpha)**

**Architecture (2 agents).**

* **Teacher (Gemini)** receives a topic and context retrieved from a static corpus using TF-IDF.  
  It **generates** a math problem, a step-by-step solution, and a final answer following a strict system prompt with the format:  
  `## PROBLEM / ## SOLUTION / ## ANSWER`.

* **Judge** verifies the Teacher’s answer using the **real Wolfram Alpha** service.  
  It translates the problem statement into a Wolfram-compatible query, sends it to the Short Answers API, receives the oracle answer, and compares it with the Teacher’s final answer.

### **Prompt and Agent Function Utils**

In [60]:
TEACHER_SYSTEM = """Ти — досвідчений учитель математики й укладач задач. Спілкуєшся виключно українською мовою.

Отримуєш ТЕМУ та КОНТЕКСТ із підручника. На їх основі:
1. Придумай одну математичну задачу саме на цю тему, спираючись на контекст.
2. Сформулюй її чітко й самодостатньо — не згадуючи відповідь в умові.
3. Розв'яжи її покроково.
4. Дай одну конкретну фінальну відповідь.

Формат відповіді (рівно ці три заголовки):
## ЗАДАЧА
[умова українською]

## РОЗВ'ЯЗОК
[послідовні кроки з поясненням]

## ВІДПОВІДЬ
[лише кінцеве значення: число або формула, без зайвих слів]

Правила:
- Складність відповідає темі; уникай надмірно громіздких обчислень.
- Відповідь має бути однозначною та перевірюваною — одне число або вираз.
- Усі формули пиши в синтаксисі коду (Python), лише символами клавіатури: + - * / ^ ( ). Замість Unicode-символів використовуй sqrt(...), x^2, a/b.
- Не додавай нічого поза цими трьома блоками."""

def parse_sections(text: str) -> dict:
    """Розбиває відповідь Teacher на ЗАДАЧА / РОЗВ'ЯЗОК / ВІДПОВІДЬ."""
    out, cur = {}, None
    for line in text.splitlines():
        m = re.match(r"#+\s*(ЗАДАЧА|РОЗВ.?ЯЗОК|ВІДПОВІДЬ)", line.strip(), re.I)
        if m:
            t = m.group(1).upper()
            cur = "ЗАДАЧА" if t.startswith("ЗАДАЧА") else ("ВІДПОВІДЬ" if t.startswith("ВІДПОВ") else "РОЗВ'ЯЗОК")
            out[cur] = ""
        elif cur is not None:
            out[cur] += line + "\n"
    return {k: v.strip() for k, v in out.items()}

def to_wolfram_query(problem_text: str) -> str:
    """Judge перекладає умову у короткий Wolfram-запит англійською."""
    sys = ("Convert the math problem into ONE short Wolfram Alpha query in English. "
           "Use only digits and + - * / ^ ( ) and words like solve/derivative/simplify. "
           "Output ONLY the query, nothing else.")
    return call_llm(TEACHER_MODEL, sys, problem_text).strip().strip("`")

def wolfram_short_answer(query: str) -> str:
    """Запит до Wolfram Alpha Short Answers API (потрібен WOLFRAM_APP_ID)."""
    app = os.getenv("WOLFRAM_APP_ID")
    if not app or not query:
        return ""
    url = "https://api.wolframalpha.com/v1/result?" + urllib.parse.urlencode({"appid": app, "i": query})
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return r.read().decode("utf-8").strip()
    except Exception as e:
        print(f"Wolfram не зміг ({query!r}): {e}")
        return ""

def answers_match(teacher: str, oracle: str) -> bool:
    """Порівняння відповіді Teacher з відповіддю Wolfram (нормалізоване, але крихке)."""
    norm = lambda s: re.sub(r"[\s'\"]+", "", (s or "").lower())
    a, b = norm(teacher), norm(oracle)
    return bool(a) and bool(b) and (a in b or b in a)

def run_baseline(topic: str) -> dict:
    ctx = get_context(topic, dynamic=False)
    hits = RETRIEVER.retrieve(topic, k=1)
    src = hits[0][1] if hits else "—"

    content = call_llm(TEACHER_MODEL, TEACHER_SYSTEM, f"Тема: {topic}\n\nКонтекст:\n{ctx}")
    sec = parse_sections(content)
    problem_text   = sec.get("ЗАДАЧА", "")
    teacher_answer = sec.get("ВІДПОВІДЬ", "")
    time.sleep(2)

    query  = to_wolfram_query(problem_text)
    oracle = wolfram_short_answer(query)
    time.sleep(2)
    verified = answers_match(teacher_answer, oracle)

    return {"topic": topic, "verified": verified, "context": src,
            "teacher_answer": teacher_answer, "query": query, "oracle": oracle}

### **Evaluation**

In [72]:
random.seed(42)

sample_topics = random.choices(TOPICS, k=18)
base = [run_baseline(t) for t in sample_topics]

base_rate = sum(b["verified"] for b in base) / len(base)
print(f"Baseline verified_rate = {base_rate:.0%}  ({sum(b['verified'] for b in base)}/{len(base)})")
for b in base:
    print("  ", "OK  " if b["verified"] else "FAIL",
          f"{b['topic']:22} | Teacher={b['teacher_answer'][:16]!r} | Wolfram={b['oracle'][:16]!r}")


Wolfram не зміг ('solve 2*(x+y)=42, x=y+9, A=x*y'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve a*x - 8 = 2*x for a where x is the smaller root of x^2 - 5*x + 6 = 0'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve 4*x + 2*a = 5*a - 3 for a where x = max(roots of x^2 - 5*x + 6 = 0)'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve 2*(x+y)=40, x=y+4, A=x*y'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve 2*(x+y)=60, x=y+10'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve 2*(x+y)=36, x=y+2, a=x*y'): HTTP Error 501: Not Implemented
Wolfram не зміг ('solve 2*(12+x)=40, A=12*x'): HTTP Error 501: Not Implemented
Baseline verified_rate = 56%  (10/18)
   FAIL Площа прямокутника     | Teacher='90' | Wolfram=''
   FAIL Лінійні рівняння       | Teacher='6' | Wolfram=''
   OK   Квадратні рівняння     | Teacher='6' | Wolfram='{w = 6}'
   OK   Квадратні рівняння     | Teacher='6' | Wolfram='the 1st is w equ'
   OK   Теорема Піфагора       | Teacher='

### **Summary**

* **Single, unspecialized agent.** One Teacher both invents the problem and answers
  it — generation and solving are never separated, so there is no independent check  on the Teacher's own work.

* **Static knowledge base.** The TF-IDF corpus is limited to the few local
  documents; covering new topics requires manually adding files.

* **Verification bottleneck (Wolfram Alpha).** The Judge depends on the Wolfram
  Short Answers API, which is sensitive to phrasing, frequently fails to parse
  multi-step queries (HTTP 501), and is compared as plain text — so equivalent but
  differently-formatted answers are wrongly rejected. Hence the low `verified_rate`.

## **Experiment 2 — Multi-Agent System (RAG + SymPy Verification)**

**Architecture improvement (3 agents):**

1. **Dynamic context (Wikipedia) — real RAG.**  
   For each topic, `get_context(topic, dynamic=True)` retrieves a relevant explanation from Wikipedia on the fly.

2. **Agent - Teacher** uses the system prompt and the retrieved context to **generate** a new math problem and its own answer using the format:  
   `## PROBLEM / ## ANSWER`.  
   This is the retrieval-augmented generation step.

3. **Agent - Solver** receives only the problem statement, solves it independently, and converts the solution into a **SymPy** expression that can be computed exactly.  
   The expected format is:  
   `## SOLUTION / ## ANSWER / ## SYMPY`.

4. **Judge** compares the Teacher’s answer with the exact result computed by SymPy and **rejects incorrect solutions**.  
   Structured data exchange between components is handled through `dataclasses`.

### **Prompt and Agent Function Utils**

In [62]:
SOLVER_SYSTEM = """Ти — уважний розв'язувач математичних задач. Спілкуєшся виключно українською мовою.

Тобі дають ЛИШЕ умову задачі (готової відповіді ти не бачиш). Розв'яжи її самостійно, крок за кроком, не вгадуючи. Окрім словесного розв'язку, переклади задачу в один вираз мовою SymPy, який обчислює точну відповідь — це гарантує відсутність арифметичних помилок.

ФОРМАТ ВІДПОВІДІ (рівно три заголовки):
## РОЗВ'ЯЗОК
[короткі послідовні кроки міркування]

## ВІДПОВІДЬ
[твоя фінальна відповідь: одне число або вираз]

## SYMPY
[рівно один рядок-вираз SymPy, що обчислює відповідь]

ПРАВИЛА ДЛЯ БЛОКУ SYMPY:
- Лише один валідний вираз, без import, без присвоєнь, без пояснень.
- Змінна — x. Доступні функції: solve, Eq, diff, integrate, sqrt, Rational, simplify, pi.
- Множення пиши як *, степінь як **. Приклади:
  solve(Eq(2*x + 3, 7), x)   diff(3*x**2 + 2*x, x)   integrate(2*x + 1, x)   Rational(3, 6)"""


with open("safe_sympy_ns.pkl", "rb") as f:
      _SP_NS = pickle.load(f)

In [63]:
@dataclass
class AgentMessage:
    sender: str
    topic: str
    statement: str
    teacher_answer: str

def eval_sympy(expr: str):
    """Обчислює вираз SymPy від Solver у безпечному просторі імен (без builtins)."""
    expr = expr.strip().strip("`")
    expr = re.sub(r"^\s*python", "", expr).strip().replace("^", "**")
    try:
        return eval(expr, _SP_NS)
    except Exception as e:
        print(f"SymPy не обчислив {expr!r}: {e}")
        return None

def values_match(teacher_text: str, oracle) -> bool:
    """Чи збігається відповідь Teacher із обчисленням SymPy (символьно)."""
    if oracle is None:
        return False
    t = teacher_text.split("=")[-1].replace("^", "**").strip().rstrip(".")
    try:
        tv = sp.sympify(t, rational=True)
    except Exception:
        return False
    cands = oracle if isinstance(oracle, (list, tuple, set)) else [oracle]
    for c in cands:
        try:
            if sp.simplify(sp.sympify(c) - tv) == 0:
                return True
        except Exception:
            pass
    return False

def teacher_generate(topic: str, n: int = 3) -> list:
    """Агент A: RAG-генерація задач із динамічного контексту Вікіпедії."""
    ctx = get_context(topic, dynamic=True)
    items = []
    for _ in range(n):
        out = call_llm(TEACHER_MODEL, TEACHER_SYSTEM, f"Тема: {topic}\n\nКонтекст:\n{ctx}")
        sec = parse_md(out)
        stmt, ans = sec.get("ЗАДАЧА", "").strip(), sec.get("ВІДПОВІДЬ", "").strip()
        if stmt and ans:
            items.append(AgentMessage("Teacher", topic, stmt, ans))
    return items

def solver_judge(msg: AgentMessage) -> dict:
    """Агент B + Judge: Solver розв'язує і дає вираз SymPy; Judge звіряє з відповіддю Teacher."""
    out = call_llm(SOLVER_MODEL, SOLVER_SYSTEM, f"Розв'яжи задачу:\n{msg.statement}")
    sec = parse_md(out)
    solution = sec.get("РОЗВ'ЯЗОК", "") or sec.get("РОЗВ’ЯЗОК", "") or "Розв'язання."
    oracle = eval_sympy(sec.get("SYMPY", ""))
    verified = values_match(msg.teacher_answer, oracle)
    output = f"{msg.statement}\n\n{solution}\nВідповідь: {msg.teacher_answer}"
    return {"input": msg.topic, "type": "task", "output": output,
            "expected_answer": msg.teacher_answer, "verified": verified}

### **Performance Examples**

In [44]:
results = teacher_generate("Похідні", 1)

print("Teacher (RAG) склав:", results[0].statement[:90])
print("Заявлена відповідь:", results[0].teacher_answer)
print("Перевірка SymPy:", solver_judge(results[0])["verified"])

Teacher (RAG) склав: Знайдіть рівняння дотичної до графіка функції f(x) = x^3 - 2*x^2 + 5 у точці з абсцисою x_
Заявлена відповідь: y = 4*x - 3
Перевірка SymPy: True


In [64]:
results = teacher_generate("Алгебра", 1)

print("Teacher (RAG) склав:", results[0].statement[:90])
print("Заявлена відповідь:", results[0].teacher_answer)
print("Перевірка SymPy:", solver_judge(results[0])["verified"])

Teacher (RAG) склав: Довжина прямокутної ділянки землі на 5 метрів більша за її ширину. Знайдіть периметр цієї 
Заявлена відповідь: 38
Перевірка SymPy: True


In [52]:
results = teacher_generate("Імовірності", 1)

print("Teacher (RAG) склав:", results[0].statement[:90])
print("Заявлена відповідь:", results[0].teacher_answer)
print("Перевірка SymPy:", solver_judge(results[0])["verified"])

Teacher (RAG) склав: У коробці лежать 5 червоних, 3 сині та 2 зелені кульки. З коробки навмання виймають одну к
Заявлена відповідь: 4/5
Перевірка SymPy: True


### **Datset Creation (Wikipedia)**

In [66]:
def build_dataset(n_per_topic: int = 3):
    rows, raw = [], []
    for topic in TOPICS:
        for msg in teacher_generate(topic, n_per_topic):
            r = solver_judge(msg)
            raw.append(r)
            if r["verified"]:
                rows.append({k: r[k] for k in ("input", "output", "type", "expected_answer")})
    return rows, raw

rows, raw = build_dataset(3)
exp2_rate = (sum(r["verified"] for r in raw) / len(raw)) if raw else 0
with open("eval_dataset.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [67]:
print(f"Generated: {len(raw)} | verified and saved: {len(rows)}")
print(f"Experiment 2 verified_rate = {exp2_rate:.0%}")

print("\nExample dataset row:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2))

Generated: 18 | verified and saved: 17
Experiment 2 verified_rate = 94%

Example dataset row:
{
  "input": "Лінійні рівняння",
  "output": "У лінійному рівнянні `a * x - b = 4` коефіцієнт `a` дорівнює сумі коренів рівняння `x^2 - 5x + 6 = 0`, а параметр `b` дорівнює їхньому добутку. Знайдіть корінь `x` цього лінійного рівняння.\n\n1. Знайдемо значення коефіцієнтів $a$ та $b$ за допомогою теореми Вієта для квадратного рівняння $x^2 - 5x + 6 = 0$:\n   - Коефіцієнт $a$ дорівнює сумі коренів: $a = 5$.\n   - Параметр $b$ дорівнює добутку коренів: $b = 6$.\n   *(Дійсно, коренями рівняння є числа $2$ та $3$, їхня сума $2 + 3 = 5$, а добуток $2 \\cdot 3 = 6$)*.\n\n2. Підставимо отримані значення $a = 5$ та $b = 6$ у початкове лінійне рівняння $a \\cdot x - b = 4$:\n   $$5 \\cdot x - 6 = 4$$\n\n3. Розв'яжемо отримане лінійне рівняння:\n   $$5x = 4 + 6$$\n   $$5x = 10$$\n   $$x = 2$$\nВідповідь: 2",
  "type": "task",
  "expected_answer": "2"
}


## **Experiment Comparison**

Metrics: **verified_rate** (the proportion of answers that passed verification).

In [73]:
consistency = sum(answer_token(r) in r["output"] for r in rows) / len(rows)
statements = [r["output"].split("\n")[0] for r in rows]
diversity = len(set(statements)) / len(statements)

cmp = pd.DataFrame([
    {"Experiment": "1. Baseline (Wolfram search)", "verified_rate": round(base_rate, 3), "context": "static corpus + TF-IDF"},
    {"Experiment": "2. Multi-Agent (SymPy)", "verified_rate": round(exp2_rate, 3), "context": "dynamic Wikipedia"},
])

display(cmp)

,Experiment,verified_rate,context
0,1. Baseline (Wolfram search),0.556,static corpus + TF-IDF
1,2. Multi-Agent (SymPy),0.944,dynamic Wikipedia


## Summary (Multi-agent + RAG + SymPy)

* **Dynamic knowledge (RAG over Wikipedia).** Context is retrieved on demand for any topic, overcoming the static-corpus limitation of baseline.

* **Roles separation.** A separate teacher model generates a task, while the another one solves it


* **Symbolic verification (deterministic)**. The Solver transforms each problem into a SymPy expression, evaluated in a sandboxed namespace; the Judge compares it symbolically to the Teacher's answer. Thus `1/2` = `0.5`, root ordering and spacing no longer lead to false rejections, and outright wrong answers are filtered out, compared to Wolfram Alpha.

* **Result**: Increased the verification rate on a test set of 18 randomly selected topics from **56%** to **94%**.